In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import numpy as np
#SEED = 45
#np.random.seed(SEED)

from models import GibbsSamplerLLFM
from data import generate_parametric
from post_process import frequency_align, hungarian_align
from sklearn.metrics import mean_squared_error


T = 500
S = 4
K = 8
alpha = 5.0
rho = 0.2
sigma_w = 1.0
mu_b = [-3.0]*S
sigma_b = 1.0

n_iter = 1000
burn = 500
n_subsample = 125
chains =4
runs = 3


def corrcoef_flat(a, b):
    return np.corrcoef(a.flatten(), b.flatten())[0,1]

In [8]:
W_nmse_list = []
b_nmse_list = []
p_nmse_list = []

W_corr_list = []
b_corr_list = []
p_corr_list = []

for run in range(runs):
    Y, Z_true, W_true, b_true, p_true = generate_parametric(T=T, S=S, K=K, alpha=alpha, rho=rho, sigma_w=sigma_w, mu_b=mu_b, sigma_b=sigma_b)
    all_Z_samples = []
    all_W_samples = []
    all_b_samples = []
    for chain in range(chains):
        sampler = GibbsSamplerLLFM(
            Data=Y,
            K=K,
            alpha=alpha,
            rho=rho,
            sigma_w=sigma_w,
            mu_b=mu_b,
            sigma_b=sigma_b,
            n_iter=n_iter,
            burn=burn,
            n_subsample=n_subsample
            )
        sampler.run()
        sampler.get_posterior_samples()
        Z_samples = sampler.good_samples_Z      # (n_samples, T, K)
        W_samples = sampler.good_samples_W      # (n_samples, K, S)
        b_samples = sampler.good_samples_b      # (n_samples, S)
        all_Z_samples.append(Z_samples)
        all_W_samples.append(W_samples)
        all_b_samples.append(b_samples)
        print(f"Chain {chain+1} completed with {Z_samples.shape[0]} samples.")

    Z_aligned_samples, W_aligned_samples, b_aligned_samples = hungarian_align(np.vstack(all_Z_samples), np.vstack(all_W_samples), np.vstack(all_b_samples), W_true)

    W_mean = W_aligned_samples.mean(axis=0)
    Z_mean = Z_aligned_samples.mean(axis=0)
    p_mean = Z_mean.mean(axis=0)
    b_mean = b_aligned_samples.mean(axis=0)
    # --- NMSE ---
    W_nmse_list.append(mean_squared_error(W_mean, W_true))
    b_nmse_list.append(mean_squared_error(b_mean, b_true))
    p_nmse_list.append(mean_squared_error(p_mean, p_true))

    # --- Correlations ---
    W_corr_list.append(corrcoef_flat(W_mean, W_true))
    b_corr_list.append(corrcoef_flat(b_mean, b_true))
    p_corr_list.append(corrcoef_flat(p_mean, p_true))
    print(f"Run {run+1}/{runs} completed.")



Chain 1 completed with 125 samples.
Chain 2 completed with 125 samples.
Chain 3 completed with 125 samples.
Chain 4 completed with 125 samples.
Run 1/3 completed.
Chain 1 completed with 125 samples.
Chain 2 completed with 125 samples.
Chain 3 completed with 125 samples.
Chain 4 completed with 125 samples.
Run 2/3 completed.
Chain 1 completed with 125 samples.
Chain 2 completed with 125 samples.
Chain 3 completed with 125 samples.
Chain 4 completed with 125 samples.
Run 3/3 completed.


In [9]:
W_nmse = np.array(W_nmse_list)
b_nmse = np.array(b_nmse_list)
p_nmse = np.array(p_nmse_list)

W_corr = np.array(W_corr_list)
b_corr = np.array(b_corr_list)
p_corr = np.array(p_corr_list)

In [10]:
print("=== NMSE ===")
print(f"W: mean={W_nmse.mean():.4f}, std={W_nmse.std():.4f}")
print(f"b: mean={b_nmse.mean():.4f}, std={b_nmse.std():.4f}")

print("\n=== Correlation ===")
print(f"W: mean={W_corr.mean():.4f}, std={W_corr.std():.4f}")
print(f"b: mean={b_corr.mean():.4f}, std={b_corr.std():.4f}")

=== NMSE ===
W: mean=0.1165, std=0.0902
b: mean=0.7250, std=0.6742

=== Correlation ===
W: mean=0.3806, std=0.4282
b: mean=0.8726, std=0.0895
